# PROJECT SENTINAL
### Isaac Feldmann


# 0. Importing python libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from matplotlib import style
style.use('ggplot')

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay



# 1. Data Ingestion and Processing

In [ ]:
# Importing the google drive from the colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Extremism_detection/elections_processed.csv')

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
#  shows first N rows
df.head(20)

In [ ]:
#  last N rows of table
df.tail()

## Data Processing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def data_processing(tweet):
    tweet = tweet.lower()
    tweet = re.sub(r"https\S+|www\S+", '', tweet, flags=re.MULTILINE)
    tweet = re.sub(r'\@w+|#', '', tweet)
    tweet = re.sub(r'[^\w\s]', '', tweet)
    tweet = re.sub(r'δ', '', tweet)
    tweet_tokens = word_tokenize(tweet)
    stop_words = set(stopwords.words('english'))
    filtered_tweets = [w for w in tweet_tokens if not w in stop_words]
    return " ".join(filtered_tweets)


In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

## Lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer

#  Defining a lemmantization
lemmatizer = WordNetLemmatizer()
def lemmatizing(data):
    #  Ensuring 'data' is a string before splitting into words
    if isinstance(data, str):
        tweet = [lemmatizer.lemmatize(word) for word in data.split()]
        return " ".join(tweet)
    else:
        return data #  return original data if not a string (shouldn't happen with current flow, but good pracitice)


In [ ]:
#  applying data processing
df['tweet'] = df['tweet'].fillna('').apply(data_processing)

In [ ]:
#  applying lemmantization function
df['tweet'] = df['tweet'].apply(lambda x:lemmatizing(x))

In [ ]:
df.head()

### Geospatial Engagement Map

In [ ]:
import plotly.express as px

def visualize_geographical_distribution(df):
    '''
    plots the spatial distribution of tweet for bot candidates.
    Uses blue for Biden and red for Trump based on lat/long coordinates
    '''
    #  ensure laat and long are numeric, coercing errors to NaN
    df['lat'] = pd.to_numeric(df['lat'], errors = 'coerce')
    df['long'] = pd.to_numeric(df['long'], errors = 'coerce')

    #  filtering for observations with the valid coordinates
    geo_df = df.dropna(subset = ['lat', 'long']).copy()

    #  creating an interactive mapbox scatter plot
    #  blue hue for Biden, Red for Trump as per political party
    fig = px.scatter_mapbox(
        geo_df,
        lat = 'lat',
        lon = 'long',
        color = 'candidate',
        color_descrete_map = {'Biden' : 'blue', 'Trump' : 'red'},
        size_max = 15,
        zoom = 3,
        mapbox_style = 'carto-position',
        title = 'Geographical Distribution of Tweets: Biden vs. Trump (2020)',
        hover_data = ['city', 'state']
    )

    fig.update_layout(margin = {'r' : 0, 't' : 40, 'l' : 0, 'b' : 0})
    fig.show()



In [ ]:
import plotly.express as px

def visualize_hate_speech_geography(df):

    hate_df = df[df('radicalization_level') == 'extermist'].copy()
    hate_geo = hate+df.dropna(subset=['lat', 'long'])

    fig = px.scatter_mapbox(
        hate_df,
        lat = 'lat',
        lon = 'long',
        color = 'candidate',
        color_descrete_map = {'Biden' : 'blue', 'Trump' : 'red'},
        size_max = 15,
        zoom = 3,
        mapbox_style = 'carto-position',
        title = 'Geographical Distribution of extermist content',
        hover_data = ['tweet', 'city', 'state']
    )

    fig.update_layout(margin = {'r' : 0, 't' : 40, 'l' : 0, 'b' : 0})
    fig.show()


## Geographic and Categorical Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_radicalization_distribution(df):
    '''
    Visualize the prevalance if normal vs. offensive vs. hate speech
    across candidate subjects.
    '''

    plt.figure(figsize=(12, 6))

    #  we use a categorical order to ensure a logical progression of radicalization
    rad_order = ['normal', 'offensive', 'extermist']

    sns.countplot(data=df, x='radicalization_level', hue='candidate',
                order=rad_order, palette='magma')

    plt.title('categorical Distribution of Radicalization Levels')
    plt.yscale('log') #  logarithmic scale to highlight minority hate speech records
    plt.xlabel('Radicalization Level')
    plt.ylabel('Log COunt of Tweets')
    plt.show()

#  radicalization distribution cell
visualize_radicalization_distribution(df)



## Data Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#  creating biagrams
vect = TfidfVectorizer(ngram_range=(1,2)).fit(df['tweet'])

In [ ]:
feature_names=vect.get_feature_names_out()
print('Number of features: {}\n'.format(len(feature_names)))
print('First 20 features:\n {}'.format(feature_names[:20]))

In [ ]:
#  creating trigrams
vect=TfidfVectorizer(ngram_range=(1,3)).fit(df['tweet'])

In [ ]:
feature_names=vect.get_feature_names_out()
print('Number of features: {}\n'.format(len(feature_names)))
print('First 20 features:\n {}'.format(feature_names[:20]))

# MACHINE LEARNING MODELS

# Train Test Split

In [ ]:
# dividing the data into two sets: x and y (columns)

x=df['tweet']
y=df['radicalization_level']
x=vect.transform(x)

In [ ]:
from sklearn.model_selection import train_test_split

#  horizontally dividing the data into train and test splits (columns)

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [ ]:
print('size of x+train', (x_train.shape))
print('size of y_train', (y_train.shape))
print('size of x_test', (x_test.shape))
print('size of y_test', (y_test.shape))

## Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline


nb_df = df.copy()
nb_df['extremist'] = nb_df['radicalization_level'].apply(lambda x: 1 if x != 'normal' else 0)

#  Prepare data for Naive Bayes model
x_nb = nb_df['tweet']
y_nb = nb_df['extremist']

#  Split the data into training and testing sets for the Naive Bayes pipeline
x_train_nb, x_test_nb, y_train_nb, y_test_nb = train_test_split(x_nb, y_nb, test_size=0.2, random_state=42)


#  tfidfvectorizer is suitable for text data
nb_pipeline = Pipeline ([
    ('tfidf', TfidfVectorizer()),
    ('mnb', MultinomialNB())
])

#  fit the pipeline on the training data
nb_pipeline.fit(x_train_nb, y_train_nb)

#  make prediction
mnb_pred = nb_pipeline.predict(x_test_nb)

#  calculate accuracy
mnb_acc = accuracy_score(mnb_pred, y_test_nb)

print('accuracy :', mnb_acc)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(mnb_pred, y_test_nb))
print('\n')
print(classification_report(mnb_pred, y_test_nb))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

#  ensure y_test_nb, x_test_nb, and nb_pipeline are defined from the Naive Bayes cell
cm = confusion_matrix(y_test_nb, nb_pipeline.predict(x_test_nb))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'extremist'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix for Naive Bayes Model')
plt.show()

## XGBoost

In [ ]:
import xgboost as xgb

#  ensure that nb_pipeline, x_train_nb, and x_test_nb are available from previous cells
#  extract the TfidfVectorizer from the nb_pipeleine
tfidf_vectorizer_for_xgboost = nb_pipeline.named_steps['tfidf']

#  transform the text data using the fitted TF - IDF vertorizzer
x_train_svc_features = tfidf_vectorizer_for_xgboost.transform(x_train_nb)
x_test_svc_features = tfidf_vectorizer_for_xgboost.transform(x_test_nb)

# initialize and train XGBoost classifier
xgb_clf = xgb.XGBClassifier(objective='binary:logistic', random_state=42)
xgb_clf.fit(x_train_svc_features, y_train_nb)

#  now you can use the xgb_clf for prediction and evaluation


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test_nb, xgb_clf.predict(x_test_svc_features), target_names=['normal', 'extremist']))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_nb, xgb_clf.predict(x_test_svc_features))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'extremist'])
disp.plot(cmap=plt.cm.Blues)
plt.title('COnfusion matrix for XGBoost model')
plt.show()

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score

#  extract the TfidfVectorizer from the nb_pipeline
tfidf_vectorizer = nb_pipeline.named_steps['tfidf']


x_train_svc_features = tfidf_vectorizer.transform(x_train_nb)
x_test_svc_features = tfidf_vectorizer.transform(x_test_nb)

svc = LinearSVC()
svc.fit(x_train_svc_features, y_train_nb)


svc_pred = svc.predict(x_test_svc_features)


svc_acc = accuracy_score(svc_pred, y_test_nb)
print('accuracy', svc_acc)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(svc_pred, y_test_nb))
print('\n')
print(classification_report(svc_pred, y_test_nb))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_nb, svc_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['normal', 'extremist'])
disp.plot(cmap=plt.cm.Blues)
plt.title('confusion matrix for SVC model')
plt.show()

# Deep Learning Models

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, BatchNormalization, SpatialDropout1D, Dense, Dropout
from tensorflow.keras.models import Sequential

def train_lstm_model(x_train_raw, y_train, x_test_raw, y_test):
    #  trains bidirectional lstm model with word embedding
    print('tokenizing and padding sequences for lstm...')

    # hyperparameters
    MAX_WORDS = 20000 # size of vocab
    MAX_LEN = 100 # max len of tweet
    EMBEDDING_DIM = 128 # size of word vectors

    tokenizer = Tokenizer(num_words=MAX_WORDS)
    tokenizer.fit_on_texts(x_train_raw)

    # convert text to sequence of integers
    x_train_seq = tokenizer.texts_to_sequences(x_train_raw)
    x_test_seq = tokenizer.texts_to_sequences(x_test_raw)

    # pad sequences to ensure uniform input shape
    x_train_pad = pad_sequences(x_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    x_test_pad = pad_sequences(x_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

    print('building lstm architechture...')
    model = Sequential()

    # embbeding layer: maps integers to dense vectors
    model.add(Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM))

    # spatial dropout drops 1D feature maps (better for embedding than standard dropout)
    model.add(SpatialDropout1D(0.2))

    # bidirectional LSTM: captures context from both directions
    model.add(Bidirectional(LSTM(64, return_sequences=True))) # return sequence for stacking
    model.add(Bidirectional(LSTM(32)))

    # dense classifier head
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.4))
    model.add(Dense(1, activation='sigmoid')) # decision layer, binary classification layer, whether something is extremist or normal

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) # Fixed loss function name

    # this section is about training
    print('training LSTM model...')
    history = model.fit( # fit means start training
        x_train_pad, y_train,
        epochs=15, # train 15 times on the data
        batch_size=64, # take 64 tweets at a time
        validation_data=(x_test_pad, y_test),
        verbose=1
    )


    return model, tokenizer, history

lstm_model, lstm_tokenizer, lstm_history = train_lstm_model(x_train_nb, y_train_nb, x_test_nb, y_test_nb)

x_test_lstm_seq = lstm_tokenizer.texts_to_sequences(x_test_nb)
x_test_lstm_pad = pad_sequences(x_test_lstm_seq, maxlen=100, padding='post', truncating='post') # Fixed laxlen to maxlen
y_pred_lstm_model = (lstm_model.predict(x_test_lstm_pad) > 0.5).astype(int).flatten()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_lstm_model_metrics_table(model, vectorizer, X_test_raw, y_test):
    """
    Generates a standalone performance dataframe for the DNN
    """
    #Preprocess (Using the same vectorizer as training)
    X_test_dense = vectorizer.transform(X_test_raw).toarray()

    #Predict (Probabilities --> Class lables)
    y_probs = model.predict(X_test_score)
    y_pred = (y_probs > 0.5).astype(int).flatten()

    #Calculate metrics
    lstm_Performance_Metrics = {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
        'Score': [
            accuracy_score(y_test, y_pred),
            precision_score(y_test, y_pred, average='weighted'),
            recall_score(y_test, y_pred, average='weighted'),
            f1_score(y_test, y_pred, average='weighted')
        ]
    }

    return pd.DataFrame(lstm_Performance_Metrics)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(classification_report(y_test_nb, y_pred_lstm_model))

In [ ]:
#Confusion Matrix
y_probs_lstm = lstm_model.predict(x_test_lstm_pad)
y_pred_lstm = (y_probs_lstm > 0.5).astype(int).flatten()

#Generate and plot confusion matrix
cm_lstm = confusion_matrix(y_test_nb, y_pred_lstm)
disp_lstm = ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=['Normal', 'Extremist'])
disp_lstm.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix for LSTM Model')
plt.show()